# **Sistema Predictivo de Delitos - Lima Metropolitana (PredictiSafe)**
### Versión Optimizada para Integración Web y Producción

**Mejoras incorporadas:**
1. **Carpeta de Exportación Unificada (`CARPETA_EXPORTACION`):** Crea y guarda todos los artefactos juntos de forma segura.
2. **One-Hot Encoding Completo (`drop_first=False`):** Cada distrito (incluyendo ANCÓN) y modalidad cuenta con su columna propia explícita.
3. **Desacoplamiento del archivo Excel:** El perfil de riesgo mensual se exporta a un archivo liviano `perfil_riesgo_mensual.json` dentro de la carpeta.
4. **Cota Inferior Estricta:** La función de predicción aplica `max(0, int(round(pred)))`, evitando denuncias negativas.
5. **Carga en Memoria Singleton:** El modelo se carga una sola vez, evaluando los 43 distritos en milisegundos para el mapa interactivo.
6. **Descarga Automática en ZIP:** Empaqueta y descarga toda la carpeta generada directamente a tu PC.

## **1. Carga de Librerías y Fuentes de Datos**

In [ ]:
# Carga de archivos en Google Colab
from google.colab import files
uploaded = files.upload()

import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración estética de gráficos
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
# ===========================================================================
# CONFIGURACIÓN DE LA CARPETA DE GUARDADO
# ===========================================================================
import os

# Nombre de la carpeta donde se guardarán TODOS los archivos generados
CARPETA_EXPORTACION = 'modelo_predictisafe_export'

# Si deseas guardarlo directamente en tu Google Drive montado, descomenta la siguiente línea:
# CARPETA_EXPORTACION = '/content/drive/MyDrive/modelo_predictisafe_export'

os.makedirs(CARPETA_EXPORTACION, exist_ok=True)
print(f"✅ Carpeta de salida lista: '{os.path.abspath(CARPETA_EXPORTACION)}'")


In [ ]:
nombre_csv = 'denunciasjan18may26.csv'
nombre_riesgo = 'dataset_riesgo_delictivo_3000.xlsx'

if nombre_csv not in uploaded:
    raise FileNotFoundError(f"No se cargó {nombre_csv}. Sube el archivo CSV.")
if nombre_riesgo not in uploaded:
    raise FileNotFoundError(f"No se cargó {nombre_riesgo}. Sube el archivo Excel.")

df_heavy = pd.read_csv(nombre_csv, sep=',')
df_riesgo = pd.read_excel(nombre_riesgo, sheet_name='Dataset_3000')

print("CSV denuncias:", df_heavy.shape)
print("Excel riesgo:", df_riesgo.shape)
display(df_heavy.head(3))

## **2. Limpieza, Corrección de Cabeceras y Filtro Lima Metropolitana**

In [ ]:
# Corrección de nombres de columnas
columnas_renombrar = {
    'ANIO': 'anio_den',
    'MES': 'mes_den',
    'DPTO_HECHO_NEW': 'dpto_hecho_new',
    'PROV_HECHO': 'prov_hecho',
    'DIST_HECHO': 'dist_hecho',
    'UBIGEO_HECHO': 'ubigeo_hecho',
    'P_MODALIDADES': 'p_modalidades',
    'cantidad': 'cant_denuncias'
}
df_heavy = df_heavy.rename(columns=columnas_renombrar)

# Filtrar únicamente Lima Metropolitana
df_lima_metro = df_heavy[
    (df_heavy['dpto_hecho_new'].str.upper() == 'LIMA METROPOLITANA') &
    (df_heavy['prov_hecho'].str.upper() == 'LIMA')
].copy()

# Limpieza de cadenas en mayúsculas sin espacios residuales
df_lima_metro['dist_hecho'] = df_lima_metro['dist_hecho'].str.strip().str.upper()
df_lima_metro['p_modalidades'] = df_lima_metro['p_modalidades'].str.strip().str.upper()

print(f"Registros en Lima Metropolitana: {len(df_lima_metro):,}")
display(df_lima_metro.head(5))

## **3. Agrupación Cronológica de Denuncias**

In [ ]:
# Agrupación por Año, Mes, Distrito y Modalidad
df_predictivo = df_lima_metro.groupby([
    'anio_den',
    'mes_den',
    'dist_hecho',
    'p_modalidades'
])['cant_denuncias'].sum().reset_index()

# Ordenar cronológicamente
df_predictivo = df_predictivo.sort_values(by=['anio_den', 'mes_den', 'dist_hecho']).reset_index(drop=True)
print("Dataset estructurado:", df_predictivo.shape)
display(df_predictivo.head(5))

## **4. Integración y Exportación del Perfil de Riesgo (Desacoplamiento del Excel)**
Calculamos los promedios mensuales de riesgo y los exportamos a `perfil_riesgo_mensual.json`. De este modo, en producción el servidor solo carga este JSON liviano en lugar de abrir el archivo Excel de 3000 filas en cada petición.

In [ ]:
variables_riesgo = ['iluminacion_deficiente_pct', 'cobertura_videovigilancia_pct']
faltantes = [c for c in ['fecha_inicio_semana'] + variables_riesgo if c not in df_riesgo.columns]
if faltantes:
    raise KeyError(f"Faltan columnas en el Excel de riesgo: {faltantes}")

df_riesgo['fecha_inicio_semana'] = pd.to_datetime(df_riesgo['fecha_inicio_semana'], errors='coerce')
df_riesgo['mes_riesgo'] = df_riesgo['fecha_inicio_semana'].dt.month

# Perfil mensual promedio
perfil_mensual_riesgo = (
    df_riesgo.groupby('mes_riesgo')[variables_riesgo]
    .mean()
    .reset_index()
    .rename(columns={'mes_riesgo': 'mes_den'})
)

df_predictivo['mes_den'] = pd.to_numeric(df_predictivo['mes_den'], errors='coerce').astype('Int64')
df_predictivo = df_predictivo.merge(perfil_mensual_riesgo, on='mes_den', how='left')

# Imputación con media global para meses sin registro directo
medias_globales = {var: float(df_riesgo[var].mean()) for var in variables_riesgo}
for variable in variables_riesgo:
    df_predictivo[variable] = df_predictivo[variable].fillna(medias_globales[variable])

# Generación del diccionario liviano para producción
perfil_riesgo_dict = {
    int(row['mes_den']): {
        'iluminacion_deficiente_pct': round(float(row['iluminacion_deficiente_pct']), 4),
        'cobertura_videovigilancia_pct': round(float(row['cobertura_videovigilancia_pct']), 4)
    }
    for _, row in perfil_mensual_riesgo.iterrows()
}
perfil_riesgo_dict['default'] = {var: round(medias_globales[var], 4) for var in variables_riesgo}

# Guardado dentro de la carpeta generada
ruta_perfil_json = os.path.join(CARPETA_EXPORTACION, 'perfil_riesgo_mensual.json')
with open(ruta_perfil_json, 'w', encoding='utf-8') as f:
    json.dump(perfil_riesgo_dict, f, indent=4)

print(f"✅ Variables integradas y perfil exportado en: '{ruta_perfil_json}'")
display(df_predictivo.head(3))


## **5. Enriquecimiento Espacial y Atributos Cíclicos Temporales**
Incorporamos coordenadas geográficas para cada distrito y transformaciones trigonométricas (`mes_sin`, `mes_cos`) para respetar la continuidad temporal.

In [ ]:
# Diccionario oficial de coordenadas de distritos de Lima Metropolitana
coordenadas_lima = {
    'ANCON': (-11.7742, -77.1758),
    'ATE': (-12.0267, -76.9189),
    'BARRANCO': (-12.1492, -77.0211),
    'BREÑA': (-12.0603, -77.0436),
    'CARABAYLLO': (-11.8500, -77.0333),
    'CHACLACAYO': (-11.9750, -76.7692),
    'CHORRILLOS': (-12.1625, -77.0219),
    'CIENEGUILLA': (-12.1000, -76.8000),
    'COMAS': (-11.9333, -77.0500),
    'EL AGUSTINO': (-12.0453, -77.0044),
    'INDEPENDENCIA': (-11.9833, -77.0500),
    'JESUS MARIA': (-12.0747, -77.0481),
    'LA MOLINA': (-12.0792, -76.9261),
    'LA VICTORIA': (-12.0653, -77.0153),
    'LIMA': (-12.0464, -77.0428),
    'LINCE': (-12.0833, -77.0333),
    'LOS OLIVOS': (-11.9667, -77.0667),
    'LURIGANCHO': (-11.9333, -76.7000),
    'LURIN': (-12.2833, -76.8667),
    'MAGDALENA DEL MAR': (-12.0911, -77.0683),
    'MIRAFLORES': (-12.1211, -77.0297),
    'PACHACAMAC': (-12.1333, -76.8667),
    'PUCUSANA': (-12.4333, -76.8000),
    'PUEBLO LIBRE': (-12.0769, -77.0631),
    'PUENTE PIEDRA': (-11.8667, -77.0833),
    'PUNTA HERMOSA': (-12.3333, -76.8333),
    'PUNTA NEGRA': (-12.3667, -76.8000),
    'RIMAC': (-12.0333, -77.0333),
    'SAN BARTOLO': (-12.3833, -76.7833),
    'SAN BORJA': (-12.1083, -77.0056),
    'SAN ISIDRO': (-12.0975, -77.0353),
    'SAN JUAN DE LURIGANCHO': (-11.9833, -77.0000),
    'SAN JUAN DE MIRAFLORES': (-12.1500, -76.9667),
    'SAN LUIS': (-12.0750, -76.9917),
    'SAN MARTIN DE PORRES': (-12.0000, -77.0833),
    'SAN MIGUEL': (-12.0778, -77.0944),
    'SANTA ANITA': (-12.0444, -76.9694),
    'SANTA MARIA DEL MAR': (-12.4000, -76.7667),
    'SANTA ROSA': (-11.8000, -77.1667),
    'SANTIAGO DE SURCO': (-12.1333, -76.9833),
    'SURQUILLO': (-12.1131, -77.0197),
    'VILLA EL SALVADOR': (-12.2167, -76.9333),
    'VILLA MARIA DEL TRIUNFO': (-12.1667, -76.9333)
}

# Mapeo de coordenadas
df_predictivo['latitud'] = df_predictivo['dist_hecho'].map(lambda d: coordenadas_lima.get(d, (-12.0464, -77.0428))[0])
df_predictivo['longitud'] = df_predictivo['dist_hecho'].map(lambda d: coordenadas_lima.get(d, (-12.0464, -77.0428))[1])

# Transformaciones cíclicas anuales (seno y coseno)
df_predictivo['mes_sin'] = np.sin(2 * np.pi * df_predictivo['mes_den'].astype(int) / 12)
df_predictivo['mes_cos'] = np.cos(2 * np.pi * df_predictivo['mes_den'].astype(int) / 12)

display(df_predictivo[['dist_hecho', 'latitud', 'longitud', 'mes_den', 'mes_sin', 'mes_cos']].head(3))

## **6. Exportación de Catálogos para el Frontend**

In [ ]:
catalogo_distritos = sorted(df_predictivo['dist_hecho'].unique().tolist())
catalogo_modalidades = sorted(df_predictivo['p_modalidades'].unique().tolist())

catalogos = {
    "distritos": catalogo_distritos,
    "modalidades": catalogo_modalidades,
    "anios_disponibles": sorted([int(a) for a in df_predictivo['anio_den'].unique()]),
    "coordenadas": {d: coordenadas_lima.get(d, (-12.0464, -77.0428)) for d in catalogo_distritos}
}

# Guardado dentro de la carpeta generada
ruta_catalogos_json = os.path.join(CARPETA_EXPORTACION, 'catalogos_frontend.json')
with open(ruta_catalogos_json, 'w', encoding='utf-8') as f:
    json.dump(catalogos, f, ensure_ascii=False, indent=4)

print(f"✅ Catálogos exportados exitosamente en: '{ruta_catalogos_json}'")


## **7. One-Hot Encoding Completo (`drop_first=False`) y Entrenamiento**
Al utilizar `drop_first=False`, cada distrito y delito cuenta con su propia columna binaria explícita. De esta manera, el primer distrito en orden alfabético (`dist_hecho_ANCON`) activa su valor en 1 de forma transparente, permitiendo que el árbol de decisión evalúe su peso directamente.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import joblib

# Aplicamos One-Hot Encoding sin omitir la primera categoría
df_features_adv = pd.get_dummies(
    df_predictivo, 
    columns=['dist_hecho', 'p_modalidades'], 
    drop_first=False, 
    dtype=int
)

# División cronológica (<= 2024 para entrenar, > 2024 para evaluar)
df_train_adv = df_features_adv[df_features_adv['anio_den'] <= 2024]
df_test_adv = df_features_adv[df_features_adv['anio_den'] > 2024]

X_train_adv = df_train_adv.drop(columns=['cant_denuncias'])
y_train_adv = df_train_adv['cant_denuncias']

X_test_adv = df_test_adv.drop(columns=['cant_denuncias'])
y_test_adv = df_test_adv['cant_denuncias']

print(f"Dimensiones de X_train (todas las columnas explícitas): {X_train_adv.shape}")
print(f"Dimensiones de X_test: {X_test_adv.shape}")

# Entrenamiento del modelo Random Forest
modelo_rf_adv = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
print("Entrenando modelo...")
modelo_rf_adv.fit(X_train_adv, y_train_adv)
print("¡Modelo entrenado con éxito!")

# Predicción y evaluación acotando a mínimo 0
pred_test = modelo_rf_adv.predict(X_test_adv)
pred_test = np.clip(pred_test, 0, None)

mae = mean_absolute_error(y_test_adv, pred_test)
r2 = r2_score(y_test_adv, pred_test)

print(f"\n--- Métricas del Modelo ---")
print(f"Error Absoluto Medio (MAE): {mae:.2f} denuncias")
print(f"Coeficiente de Determinación (R²): {r2:.4f}")

# Guardado dentro de la carpeta generada
ruta_modelo_joblib = os.path.join(CARPETA_EXPORTACION, 'modelo_prediccion_delitos_lima_enriquecido.joblib')
ruta_columnas_joblib = os.path.join(CARPETA_EXPORTACION, 'columnas_modelo.joblib')

columnas_modelo = X_train_adv.columns.tolist()
joblib.dump(modelo_rf_adv, ruta_modelo_joblib)
joblib.dump(columnas_modelo, ruta_columnas_joblib)

print(f"\n✅ Artefactos guardados en '{CARPETA_EXPORTACION}':")
print(f" - '{ruta_modelo_joblib}'")
print(f" - '{ruta_columnas_joblib}'")


## **8. Función de Predicción de Producción Optimizada**
* Carga los artefactos en memoria **una sola vez** (no en cada llamada).
* Usa el diccionario precalculado de riesgo mensual.
* Soporta todos los distritos y modalidades de forma unívoca (`drop_first=False`).
* Garantiza **cota inferior no negativa** con `max(0, int(round(pred)))`.
* Devuelve el nivel de severidad (`critical`, `high`, `medium`, `low`) para colorear el mapa de Leaflet.

In [ ]:
# Carga en memoria única desde la carpeta generada
ruta_modelo_joblib = os.path.join(CARPETA_EXPORTACION, 'modelo_prediccion_delitos_lima_enriquecido.joblib')
ruta_columnas_joblib = os.path.join(CARPETA_EXPORTACION, 'columnas_modelo.joblib')
ruta_perfil_json = os.path.join(CARPETA_EXPORTACION, 'perfil_riesgo_mensual.json')

MODELO_CARGADO = joblib.load(ruta_modelo_joblib)
COLUMNAS_CARGADAS = joblib.load(ruta_columnas_joblib)

with open(ruta_perfil_json, 'r', encoding='utf-8') as f:
    PERFIL_RIESGO_MEMORIA = json.load(f)

def predecir_delitos_avanzado(
    anio: int,
    mes: int,
    distrito: str,
    modalidad: str,
    modelo=MODELO_CARGADO,
    columnas=COLUMNAS_CARGADAS,
    perfil_riesgo=PERFIL_RIESGO_MEMORIA
):
    # 1. Registro base con ceros
    X_input = pd.DataFrame(0, index=[0], columns=columnas)

    # 2. Variables temporales y cíclicas
    X_input['anio_den'] = int(anio)
    X_input['mes_den'] = int(mes)
    X_input['mes_sin'] = np.sin(2 * np.pi * mes / 12)
    X_input['mes_cos'] = np.cos(2 * np.pi * mes / 12)

    # 3. Variables de riesgo desde el diccionario precargado
    info_mes = perfil_riesgo.get(str(mes), perfil_riesgo.get(int(mes), perfil_riesgo.get('default', {})))
    if 'iluminacion_deficiente_pct' in X_input.columns:
        X_input['iluminacion_deficiente_pct'] = info_mes.get('iluminacion_deficiente_pct', 25.0)
    if 'cobertura_videovigilancia_pct' in X_input.columns:
        X_input['cobertura_videovigilancia_pct'] = info_mes.get('cobertura_videovigilancia_pct', 50.0)

    # 4. Asignación de coordenadas del distrito
    dist_upper = distrito.strip().upper()
    mod_upper = modalidad.strip().upper()
    coords = coordenadas_lima.get(dist_upper, (-12.0464, -77.0428))
    X_input['latitud'] = coords[0]
    X_input['longitud'] = coords[1]

    # 5. One-Hot Encoding explícito (todas las categorías existen con drop_first=False)
    col_distrito = f"dist_hecho_{dist_upper}"
    col_modalidad = f"p_modalidades_{mod_upper}"
    if col_distrito in X_input.columns:
        X_input[col_distrito] = 1
    if col_modalidad in X_input.columns:
        X_input[col_modalidad] = 1

    # 6. Predicción acotada a no negativos y redondeada a enteros
    pred_raw = modelo.predict(X_input)[0]
    denuncias_estimadas = max(0, int(round(pred_raw)))

    # 7. Nivel de riesgo para el frontend
    if denuncias_estimadas >= 150:
        nivel_riesgo = 'critical'
    elif denuncias_estimadas >= 80:
        nivel_riesgo = 'high'
    elif denuncias_estimadas >= 35:
        nivel_riesgo = 'medium'
    else:
        nivel_riesgo = 'low'

    return {
        "anio": int(anio),
        "mes": int(mes),
        "distrito": dist_upper,
        "modalidad": mod_upper,
        "prediccion_denuncias": denuncias_estimadas,
        "nivel_riesgo": nivel_riesgo,
        "coordenadas": {
            "latitud": coords[0],
            "longitud": coords[1]
        }
    }

# Casos de prueba: ANCON (primero alfabético) y MIRAFLORES
test_ancon = predecir_delitos_avanzado(2026, 12, 'ANCON', 'ROBO')
test_miraflores = predecir_delitos_avanzado(2026, 12, 'MIRAFLORES', 'ROBO')

print("Resultado ANCON:", json.dumps(test_ancon, indent=2))
print("\nResultado MIRAFLORES:", json.dumps(test_miraflores, indent=2))


## **9. Simulación de Mapa de Calor Predictivo Masivo (43 Distritos)**

In [ ]:
def obtener_mapa_calor_predictivo(anio: int, mes: int, modalidad: str):
    """Genera la predicción masiva en milisegundos para los 43 distritos."""
    predicciones_zonas = []
    for distrito in catalogo_distritos:
        res = predecir_delitos_avanzado(anio, mes, distrito, modalidad)
        predicciones_zonas.append({
            "Distrito": res["distrito"],
            "Prediccion_Denuncias": res["prediccion_denuncias"],
            "Nivel_Riesgo": res["nivel_riesgo"],
            "Latitud": res["coordenadas"]["latitud"],
            "Longitud": res["coordenadas"]["longitud"]
        })

    df_mapa = pd.DataFrame(predicciones_zonas)
    df_mapa = df_mapa.sort_values(by="Prediccion_Denuncias", ascending=False).reset_index(drop=True)
    return df_mapa

# Prueba masiva para Diciembre de 2026 en ROBO
df_mapa_2026 = obtener_mapa_calor_predictivo(2026, 12, "ROBO")
print("Top 10 Distritos con mayor predicción de ROBOS:")
display(df_mapa_2026.head(10))

# Gráfico de distribución de riesgo
plt.figure(figsize=(12, 6))
sns.barplot(
    data=df_mapa_2026.head(15),
    x="Prediccion_Denuncias",
    y="Distrito",
    palette="Reds_r",
    hue="Distrito",
    legend=False
)
plt.title("Top 15 Distritos con Mayor Riesgo de Robo Predicho (Diciembre 2026)", fontsize=14)
plt.xlabel("Denuncias Estimadas", fontsize=12)
plt.ylabel("Distrito", fontsize=12)
plt.show()

## **10. Conexión Directa con el Backend FastAPI**
Descarga los siguientes tres archivos desde Colab y colócalos en tu proyecto `serenazgo-frontend`:
1. `modelo_prediccion_delitos_lima_enriquecido.joblib` -> en `app/ml_models/`
2. `columnas_modelo.joblib` -> en `app/ml_models/`
3. `perfil_riesgo_mensual.json` -> en `app/ml_models/`
4. `catalogos_frontend.json` -> en `src/data/`

Con esta estructura, el backend responderá en menos de 10 milisegundos a las peticiones del frontend en React.

## **11. Empaquetado y Descarga de la Carpeta Generada**
Esta celda comprueba que los 4 archivos existan dentro de la carpeta y genera un archivo `.zip` para descargarlo automáticamente a tu computadora.

In [ ]:
# ===========================================================================
# VERIFICACIÓN Y DESCARGA AUTOMÁTICA DE LA CARPETA GENERADA
# ===========================================================================
import os
import shutil
from google.colab import files

archivos_esperados = [
    'modelo_prediccion_delitos_lima_enriquecido.joblib',
    'columnas_modelo.joblib',
    'perfil_riesgo_mensual.json',
    'catalogos_frontend.json'
]

print(f"=== VERIFICANDO ARCHIVOS EN '{CARPETA_EXPORTACION}' ===")
archivos_en_carpeta = os.listdir(CARPETA_EXPORTACION)
todos_ok = True
for arch in archivos_esperados:
    if arch in archivos_en_carpeta:
        peso_kb = os.path.getsize(os.path.join(CARPETA_EXPORTACION, arch)) / 1024
        print(f"  [OK] {arch} ({peso_kb:.1f} KB)")
    else:
        print(f"  [FALTA] {arch}")
        todos_ok = False

if todos_ok:
    # 1. Comprimir toda la carpeta en un solo archivo ZIP
    nombre_zip = 'modelo_predictisafe_export'
    shutil.make_archive(nombre_zip, 'zip', CARPETA_EXPORTACION)
    archivo_zip_final = f"{nombre_zip}.zip"
    print(f"\n✅ Carpeta comprimida exitosamente en: '{archivo_zip_final}'")
    
    # 2. Descargar automáticamente el ZIP a tu computadora
    print("⬇️ Iniciando descarga automática a tu PC...")
    files.download(archivo_zip_final)
else:
    print("\n⚠️ Algunos archivos no se generaron. Revisa las celdas anteriores.")
